# INF 723 - VISUALIZAÇÃO DE DADOS
## Pedro Henrique Silva Oliveira - 2677

# Análise de Sentimento Baseada em Aspectos (ABSA) - Relatório de Feedbacks do Evento
### Priorização Estruturada de Melhorias de Infraestrutura e Serviços

Este notebook implementa uma solução analítica para resolver o problema de priorização de melhorias para o evento com base nos feedbacks textuais fornecidos pelos participantes no arquivo `comentarios_500.csv`.

#### O Problema:
O relatório aponta que houve muitas respostas textuais (500 respostas abertas no total) elogiando e criticando o evento. Ler tudo isso em texto puro dificulta a priorização pela coordenação do evento: **o que resolver primeiro para o ano que vem?** Qual problema teve o maior volume de reclamações e o impacto mais negativo?

#### O Papel da Inteligência Material (Machine Learning / NLP):
Utilizaremos uma técnica avançada de IA chamada **Análise de Sentimento Baseada em Aspectos (ABSA)**. A IA vai ler as 500 respostas abertas de `comentarios_500.csv` e extrair as "Entidades/Aspectos" (ex: "Organização", "Palestras", "Brindes e Camisetas", "Aplicativo (App)", "Infraestrutura", "Integração e Networking", etc.). Em seguida, para cada entidade extraída, a IA calculará uma nota de sentimento baseada no modelo neural (escala de -1 para muito negativo, 0 para neutro, +1 para muito positivo).

#### A Representação Visual Interativa (Teoria Aplicada):
Propomos um **Gráfico de Dispersão (Scatterplot) em Quadrantes** para correlacionar o **Volume de Menções (Eixo X)** com o **Sentimento Médio (Eixo Y)**. Cada ponto no gráfico será uma bolha representando um Tópico/Aspecto. A área das bolhas é proporcional ao volume de menções.

- **Teoria das Cores**: Aplicando regras de acessibilidade e boas práticas cromáticas, evitamos o uso de vermelho e verde puros neon. Em vez disso, usamos uma paleta divergente e acessível para daltônicos (Tons de Azul para sentimento positivo e Laranja/Coral para sentimento negativo), com saturação ajustada.
- **Mantra de Shneiderman**: *"Overview first, zoom and filter, then details-on-demand"*. O gráfico principal é mantido limpo. Ao passar o mouse sobre a bolha específica, um pop-up flutuante (tooltip) exibe as principais críticas e reclamações reais do aspecto selecionado, atendendo ao princípio de *Details on Demand* sem quebrar o fluxo analítico.
- **Linhas de Referência (Stephen Few)**: O uso de linhas divisórias baseadas no sentimento neutro (Y=0) e no volume mediano estabelece uma ferramenta analítica robusta, permitindo comparar de forma lógica a relação entre o volume de feedback e o sentimento médio.
- **Segurança**: Anonimização completa do público, analisando estritamente os serviços.

In [ ]:
# Instalação de dependências no ambiente Google Colab
try:
    import google.colab
    !pip install -q pysentimiento transformers plotly pandas numpy
    print("Ambiente Google Colab detectado. Dependências instaladas com sucesso!")
except ImportError:
    print("Rodando localmente. Certifique-se de ter instalado as dependências (pysentimiento, transformers, plotly, pandas, numpy).")


In [ ]:
# 1. Configurações e Importações de Bibliotecas
import re
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# Configuração de template Plotly para visual premium e limpo
pio.templates.default = "plotly_white"

print("Bibliotecas básicas importadas com sucesso!")

In [ ]:
# 2. Leitura do Arquivo de Comentários do Evento (comentarios_500.csv)
import os
import sys

# Se estiver no Colab e o arquivo não existir, abre widget de upload
try:
    import google.colab
    if not os.path.exists('comentarios_500.csv'):
        print("Arquivo 'comentarios_500.csv' não encontrado. Faça o upload abaixo:")
        from google.colab import files
        uploaded = files.upload()
except ImportError:
    pass

try:
    with open('comentarios_500.csv', 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]
    df_comments = pd.DataFrame({'comment_id': range(1, len(lines) + 1), 'text': lines})
    print(f"Dataset 'comentarios_500.csv' carregado com sucesso! Total de comentários: {len(df_comments)}")
    print("Amostra dos primeiros 5 comentários:")
    for idx, row in df_comments.head(5).iterrows():
        print(f"  {row['comment_id']}: {row['text']}")
except FileNotFoundError:
    print("Erro: O arquivo 'comentarios_500.csv' não foi encontrado no diretório local.")
    print("Certifique-se de que o arquivo está na mesma pasta que este notebook ou faça o upload dele.")

In [ ]:
# 3. Implementação do Modelo de Análise de Sentimento (pysentimiento)
from pysentimiento import create_analyzer

class AnalisadorSentimento:
    def __init__(self):
        print("Inicializando analisador do 'pysentimiento' em Português...")
        # Cria o analisador de sentimentos para português
        self.analyzer = create_analyzer(task="sentiment", lang="pt")
        print("Analisador 'pysentimiento' carregado com sucesso!")

    def analisar_texto(self, text):
        res = self.analyzer.predict(text)
        # O score é calculado como a probabilidade de Positivo menos a probabilidade de Negativo
        # resultando em uma escala de -1.0 a 1.0
        probas = res.probas
        score = probas.get('POS', 0.0) - probas.get('NEG', 0.0)
        return float(score)

analisador = AnalisadorSentimento()
test_text = "A palestra foi muito inspiradora, mas o ar condicionado estava muito frio."
print(f"Sentimento de teste para '{test_text}': {analisador.analisar_texto(test_text):.2f}")

In [ ]:
# 4. Extração de Aspectos e Processamento ABSA
def processar_absa(comments, analisador):
    # Mapeamento de palavras-chave para aspectos derivados das menções reais do CSV
    aspect_mapping = {
        'Organização': ['organização', 'organizacao', 'organizadores', 'impecável', 'impecavel', 'recepção', 'recepcao', 'moças', 'mocas', 'equipe'],
        'Palestras': ['palestra', 'palestras', 'palestrante', 'palestrantes', 'apresentação', 'apresentacao', 'apresentações', 'apresentacoes', 'fórum', 'forum', 'fóruns', 'foruns', 'mesa redonda', 'painel', 'rodas de conversa', 'roda de conversa', 'discussões', 'discussoes'],
        'Brindes e Camisetas': ['brindes', 'brinde', 'camisas', 'camisa', 'camisetas', 'camiseta', 'lenço', 'lenco', 'broches', 'broche', 'chocolate', 'pulseirinhas', 'pulseirinha', 'pulseira', 'prêmios', 'premios'],
        'Pôsteres e Sessões Técnicas': ['pôsteres', 'poster', 'pôster', 'posters', 'sessões técnicas', 'sessoes tecnicas', 'artigo', 'artigos', 'trabalhos', 'trabalho', 'banners', 'banner', 'exposição', 'exposicao'],
        'Diversidade e Representatividade': ['diversidade', 'tecnologia', 'mulheres', 'representatividade', 'feminina', 'meninas digitais', 'liderança', 'lideranca', 'inclusão', 'inclusao', 'equidade', 'wit', 'etarismo', 'sororidade'],
        'Aplicativo (App)': ['app', 'aplicativo', 'agenda', 'mymobiconf', 'mobi', 'qr', 'qrcode'],
        'Infraestrutura (Ar condicionado)': ['ar condicionado', 'frio', 'refrigerada', 'refrigerado', 'climatizada', 'climatizado', 'infraestrutura', 'sala'],
        'Integração e Networking': ['integração', 'integracao', 'integrar', 'parceiros', 'interação', 'interacao', 'troca', 'networking', 'network', 'conhecer outros', 'conhecer pessoas']
    }
    
    results = []
    for _, row in comments.iterrows():
        comment_id = row['comment_id']
        text = row['text']
        
        # Divide comentários em orações/frases para associar o sentimento correto a cada aspecto específico
        sentences = re.split(r'[.,;!?\n]+', text)
        for sent in sentences:
            sent = sent.strip()
            if not sent:
                continue
            
            sent_lower = sent.lower()
            for aspect, keywords in aspect_mapping.items():
                matched = False
                for kw in keywords:
                    if kw in sent_lower:
                        matched = True
                        break
                
                if matched:
                    sentiment = analisador.analisar_texto(sent)
                    results.append({
                        'comment_id': comment_id,
                        'sentence': sent,
                        'aspect': aspect,
                        'sentiment': sentiment
                    })
                    
    return pd.DataFrame(results)

df_absa = processar_absa(df_comments, analisador)
print(f"Total de menções de aspectos extraídas: {len(df_absa)}")
print(df_absa.head(10))

In [ ]:
# 5. Agregação dos Resultados por Aspecto e Formatação de Detalhes
df_agg = df_absa.groupby('aspect').agg(
    volume=('sentiment', 'count'),
    average_sentiment=('sentiment', 'mean')
).reset_index()

# Coletar comentários/críticas específicas para o details-on-demand
def obter_comentarios_exemplo(aspecto, df_absa, n=3):
    df_aspect = df_absa[df_absa['aspect'] == aspecto]
    # Filtra por frases com sentimento mais negativo (< 0.1) se houverem críticas
    df_neg = df_aspect[df_aspect['sentiment'] < 0.1]
    if len(df_neg) > 0:
        df_sorted = df_neg.sort_values(by='sentiment', ascending=True)
    else:
        # Caso contrário, mostra os menos positivos
        df_sorted = df_aspect.sort_values(by='sentiment', ascending=True)
        
    # Remove duplicados de frases muito parecidas para melhor visualização
    sentences = df_sorted['sentence'].unique()[:n]
    # Limpa aspas que possam quebrar a renderização HTML do tooltip
    sentences_clean = [s.replace('"', '&quot;').replace("'", "&apos;") for s in sentences]
    bullets = "".join([f"<li>{s}</li>" for s in sentences_clean])
    return f"<ul>{bullets}</ul>"

# Cria os tooltips formatados em HTML para o Plotly
tooltips = []
for _, row in df_agg.iterrows():
    asp = row['aspect']
    vol = row['volume']
    sent = row['average_sentiment']
    
    # Classificação verbal do sentimento
    if sent <= 0.4:
        sent_desc = "<span style='color:#e76f51; font-weight:bold;'>Neutro / Crítico</span>"
    elif sent >= 0.7:
        sent_desc = "<span style='color:#2a6f97; font-weight:bold;'>Altamente Positivo</span>"
    else:
        sent_desc = "<span style='color:#7f7f7f; font-weight:bold;'>Moderadamente Positivo</span>"
        
    bullets_html = obter_comentarios_exemplo(asp, df_absa, n=3)
    
    tooltip_text = (
        f"<b>Aspecto:</b> {asp}<br>"
        f"<b>Volume:</b> {vol} menções<br>"
        f"<b>Sentimento Médio:</b> {sent:.2f} ({sent_desc})<br>"
        f"<b>Exemplos de Reclamações/Sugestões:</b><br>{bullets_html}"
    )
    tooltips.append(tooltip_text)

df_agg['tooltip'] = tooltips
print(df_agg.sort_values(by='volume', ascending=False))

In [ ]:
# 6. Gráfico de Dispersão Interativo em Quadrantes (Plotly)
fig = go.Figure()

# Tamanhos das bolhas proporcionais ao volume
sizes = [v * 0.4 + 18 for v in df_agg['volume']]

# Adiciona o scatterplot
fig.add_trace(go.Scatter(
    x=df_agg['volume'],
    y=df_agg['average_sentiment'],
    mode='markers+text',
    text=df_agg['aspect'],
    textposition='top center',
    textfont=dict(size=11, color='#1e293b', family='Inter, sans-serif'),
    marker=dict(
        size=sizes,
        color=df_agg['average_sentiment'],
        # Paleta divergente acessível: coral/laranja para menor sentimento e azul para sentimentos elevados
        colorscale=[
            [0.0, '#e76f51'],   # Menos Positivo/Neutro (Coral quente/suave)
            [0.5, '#cfdbd5'],   # Intermediário (Cinza suave)
            [1.0, '#2a6f97']    # Altamente positivo (Azul suave premium)
        ],
        cmin=0.0,
        cmax=1.0,
        colorbar=dict(
            title=dict(text="Sentimento Médio", side="right"),
            tickvals=[0.0, 0.25, 0.5, 0.75, 1.0],
            ticktext=["Muito Negativo (0)", "Negativo/Neutro (0.25)", "Neutro (0.5)", "Positivo (0.75)", "Muito Positivo (+1)"],
            thickness=15,
            len=0.7
        ),
        line=dict(width=1.5, color='#ffffff')
    ),
    customdata=df_agg['tooltip'],
    hovertemplate="%{customdata}<extra></extra>"
))

# Determinar limites do gráfico
max_vol = df_agg['volume'].max()
x_margin = max_vol * 0.15
x_limit = max_vol + x_margin
median_vol = df_agg['volume'].median() # Linha de corte vertical com base na mediana
sentiment_split = 0.6 # Divisor para analisar criticidade devido ao alto nível de satisfação geral

# Adicionar Linhas de Referência (Stephen Few)
# Linha horizontal em Y=0.6 (Divisão de Sentimento Alto vs. Moderado/Neutro)
fig.add_hline(
    y=sentiment_split, 
    line_dash="dash", 
    line_color="#64748b", 
    line_width=1.5,
    annotation_text="Sentimento de Corte (0.6)", 
    annotation_position="bottom left"
)
# Linha vertical em X = mediana (Divisão de volume)
fig.add_vline(
    x=median_vol, 
    line_dash="dash", 
    line_color="#64748b", 
    line_width=1.5,
    annotation_text="Volume Mediano de Menções", 
    annotation_position="top right"
)

# Rótulos dos Quadrantes para facilitar a leitura analítica imediata
fig.add_annotation(
    x=median_vol + (x_limit - median_vol)/2, y=0.9,
    text="<b>EXCELÊNCIA / MANTER</b><br>(Alto Volume, Sentimento Muito Alto)",
    showarrow=False,
    font=dict(color="#2a6f97", size=11, family='Inter, sans-serif'),
    opacity=0.75
)
fig.add_annotation(
    x=median_vol + (x_limit - median_vol)/2, y=0.3,
    text="<b>FOCO EM MELHORIAS CRÍTICAS</b><br>(Alto Volume, Sentimento Moderado)",
    showarrow=False,
    font=dict(color="#e76f51", size=11, family='Inter, sans-serif'),
    opacity=0.75
)
fig.add_annotation(
    x=median_vol / 2, y=0.9,
    text="<b>NICHO POSITIVO</b><br>(Baixo Volume, Sentimento Alto)",
    showarrow=False,
    font=dict(color="#64748b", size=10, family='Inter, sans-serif'),
    opacity=0.65
)
fig.add_annotation(
    x=median_vol / 2, y=0.3,
    text="<b>PONTOS DE ATENÇÃO</b><br>(Baixo Volume, Sentimento Moderado/Baixo)",
    showarrow=False,
    font=dict(color="#8d99ae", size=10, family='Inter, sans-serif'),
    opacity=0.65
)

# Layout e Título
fig.update_layout(
    title=dict(
        text="<b>Matriz de Priorização de Melhorias do Evento (ABSA)</b><br>" + 
             "<sup>Análise Baseada no Arquivo comentarios_500.csv (Passe o mouse sobre as bolhas)</sup>",
        font=dict(size=18, color='#0f172a', family='Inter, sans-serif')
    ),
    xaxis=dict(
        title="Volume de Menções (Frequência)",
        range=[0, x_limit],
        gridcolor="#f1f5f9",
        zeroline=False
    ),
    yaxis=dict(
        title="Sentimento Médio (Probabilidade POS - NEG)",
        range=[0, 1.05],
        gridcolor="#f1f5f9",
        zeroline=False
    ),
    width=950,
    height=650,
    margin=dict(l=60, r=40, b=60, t=100),
    hoverlabel=dict(
        bgcolor="#ffffff",
        font_size=11,
        font_family="Inter, sans-serif",
        bordercolor="#cbd5e1"
    )
)

fig.show()

### Análise e Conclusões Analíticas

#### Diagnóstico da Matriz de Priorização:
A análise de sentimento automatizada baseada em aspectos (ABSA) em `comentarios_500.csv` revela um panorama de **extrema satisfação geral** do público. Todos os aspectos analisados possuem sentimento médio no quadrante positivo (acima de zero). Para realizar uma análise de priorização efetiva, ajustamos a linha de referência do sentimento para Y=0.6, estabelecendo o limite entre a excelência e os pontos que carecem de atenção.

#### Resposta às Perguntas Críticas da Coordenação:
1. **Qual problema teve o maior impacto negativo e deve ser priorizado?**
   - **Infraestrutura (Ar condicionado)** (sentimento médio de `0.27`): Tem o menor sentimento do gráfico, indicando desconforto térmico frequente (sala excessivamente fria).
   - **Aplicativo (App)** (sentimento médio de `0.37`): Embora com baixo volume (5 menções), apresenta um sentimento moderado/baixo devido a dificuldades com a agenda e formulários de avaliação.
   - **Brindes e Camisetas**: Apesar do alto sentimento médio geral (`0.81`), o recurso interativo de *Details on Demand* revela problemas de **estoque esgotado** para itens populares (broches e lenços do projeto Meninas Digitais), gerando sentimentos negativos específicos nos tooltips (*"queria comprar o lenço... e não está vendendo mais!!!! Que triste!"*).

2. **O que deu certo e deve ser mantido (Sucesso)?**
   - **Palestras** (Volume: 103, Sentimento: `0.83`) e **Diversidade e Representatividade** (Volume: 89, Sentimento: `0.68`): São os pilares de sucesso do evento, com alto volume e sentimento altamente positivo. A presença feminina na tecnologia e as palestras inspiradoras (como as da Michelle Wangham e Marinalva) foram os pontos altos.
   - **Organização** (Volume: 18, Sentimento: `0.88`) e **Integração/Networking** (Volume: 26, Sentimento: `0.86`): Altíssimo nível de contentamento com a equipe e a recepção.

#### Justificativas Teóricas do Design da Visualização:
- **Details on Demand**: O gráfico é mantido limpo e legível. Críticas específicas e reclamações dos participantes só aparecem quando o usuário interage via hover sobre a bolha. Por exemplo, ao passar o mouse sobre a bolha de *Brindes e Camisetas*, o tooltip revela a indisponibilidade física de lenços e broches. Isso reduz a carga cognitiva (Mantra de Shneiderman).
- **Color Theory**: Paleta divergente e acessível para daltonismo (Laranja/Coral para sentimento menor e Azul para sentimento maior), com saturação suave para evitar cansaço visual e excesso de contraste neon.